In [2]:
import requests
from bs4 import BeautifulSoup
import json

def extract_east_asian_urls(main_url):
    """
    Extracts all sub-cuisine links from the 'East Asian cuisine' 
    section of the List of Cuisines Wikipedia page.
    """
    # 1. Fetch the page
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AI-Coursework-Bot/1.0"}
    response = requests.get(main_url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to retrieve page. Status code: {response.status_code}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    
    # 2. Locate the East Asian section
    # Based on your HTML screenshot, it's an <h4> inside a div.mw-heading4
    heading_id = soup.find(id="East_Asian_cuisine")
    if not heading_id:
        print("Could not find the 'East Asian cuisine' section.")
        return []

    # 3. Find the container (div-col) that follows this heading
    # We move up to the parent div then look for the sibling div-col
    parent_heading_div = heading_id.find_parent('div', class_='mw-heading')
    link_container = parent_heading_div.find_next_sibling('div', class_='div-col')

    if not link_container:
        print("Could not find the link container (div-col) after the heading.")
        return []

    # 4. Extract links from the <ul> list items
    found_links = []
    for li in link_container.find_all('li'):
        anchor = li.find('a', href=True)
        if anchor:
            title = anchor.get_text().strip()
            path = anchor['href']
            
            # Ensure it's a valid internal wiki link and not a meta page
            if path.startswith('/wiki/') and ':' not in path:
                full_url = f"https://en.wikipedia.org{path}"
                found_links.append({
                    "cuisine_name": title,
                    "url": full_url
                })

    return found_links

    

In [3]:
WIKI_LIST_URL = "https://en.wikipedia.org/wiki/List_of_cuisines"
    
print(f"Starting extraction from: {WIKI_LIST_URL}")
east_asian_urls = extract_east_asian_urls(WIKI_LIST_URL)

print(f"Successfully found {len(east_asian_urls)} cuisine links.")

# Save the output for the next module
with open('east_asian_links.json', 'w', encoding='utf-8') as f:
    json.dump(east_asian_urls, f, indent=4, ensure_ascii=False)

for link in east_asian_urls[:5]: # Preview first 5
    print(f" - {link['cuisine_name']}: {link['url']}")

Starting extraction from: https://en.wikipedia.org/wiki/List_of_cuisines
Successfully found 22 cuisine links.
 - Japanese cuisine: https://en.wikipedia.org/wiki/Japanese_cuisine
 - Ainu cuisine: https://en.wikipedia.org/wiki/Ainu_cuisine
 - Kaiseki: https://en.wikipedia.org/wiki/Kaiseki
 - Okinawan cuisine: https://en.wikipedia.org/wiki/Okinawan_cuisine
 - Japanese regional cuisine: https://en.wikipedia.org/wiki/Japanese_regional_cuisine


In [13]:
def get_links_from_page(url, headers):
    """General purpose link extractor for both main page and category pages."""
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    links = []
    
    # Check 1: Look for CategoryTree items (for the main landing page)
    tree_items = soup.find_all('div', class_='CategoryTreeItem')
    for item in tree_items:
        a = item.find('a', href=True)
        if a:
            links.append({'title': a.text.strip(), 'url': f"https://en.wikibooks.org{a['href']}"})
            
    # Check 2: Look for standard Wiki Category members (for the sub-pages)
    # Wikibooks category pages list recipes in a div with id 'mw-pages'
    category_div = soup.find('div', id='mw-pages')
    if category_div:
        for a in category_div.find_all('a', href=True):
            links.append({'title': a.text.strip(), 'url': f"https://en.wikibooks.org{a['href']}"})
    
    return links

def get_sub_links():
    BASE_URL = "https://en.wikibooks.org/wiki/Cookbook:East_Asian_cuisines"
    headers = {"User-Agent": "AI-Coursework-Bot/1.0"}
    
    print("🔍 Scanning Base Page...")
    initial_items = get_links_from_page(BASE_URL, headers)
    print(initial_items)
    
    final_recipes = []
    seen_urls = set()

    for item in initial_items:
        # If it's a 'folder' (Category)
        if "Category:" in item['url']:
            print(f"📂 Opening Folder: {item['title']}")
            sub_recipes = get_links_from_page(item['url'], headers)
            for sub in sub_recipes:
                if "Category:" not in sub['url'] and sub['url'] not in seen_urls:
                    print({"name": sub['title'], "url": sub['url'], "origin": item['title']})
                    final_recipes.append({"name": sub['title'], "url": sub['url'], "origin": item['title']})
                    seen_urls.add(sub['url'])
            time.sleep(0.5)
            
        # If it's a direct recipe link
        else:
            if item['url'] not in seen_urls:
                print(f"📄 Grabbing Direct: {item['title']}")
                final_recipes.append({"name": item['title'], "url": item['url'], "origin": "Main Page"})
                seen_urls.add(item['url'])

    # Write to JSON
    with open('wikibooks_complete.json', 'w', encoding='utf-8') as f:
        json.dump(final_recipes, f, indent=4, ensure_ascii=False)
    
    print(f"\n✅ Total Recipes Harvested: {len(final_recipes)}")


get_sub_links()

🔍 Scanning Base Page...
[{'title': 'Chinese recipes', 'url': 'https://en.wikibooks.org/wiki/Category:Chinese_recipes'}, {'title': 'Japanese recipes', 'url': 'https://en.wikibooks.org/wiki/Category:Japanese_recipes'}, {'title': 'Korean recipes', 'url': 'https://en.wikibooks.org/wiki/Category:Korean_recipes'}, {'title': 'Taiwanese recipes', 'url': 'https://en.wikibooks.org/wiki/Category:Taiwanese_recipes'}, {'title': 'Broccoli Stir Fry', 'url': 'https://en.wikibooks.org/wiki/Cookbook:Broccoli_Stir_Fry'}, {'title': 'Cream Cheese Wontons', 'url': 'https://en.wikibooks.org/wiki/Cookbook:Cream_Cheese_Wontons'}, {'title': 'Fried Rice', 'url': 'https://en.wikibooks.org/wiki/Cookbook:Fried_Rice'}, {'title': 'Onigiri', 'url': 'https://en.wikibooks.org/wiki/Cookbook:Onigiri'}, {'title': 'Spicy Miso Udon', 'url': 'https://en.wikibooks.org/wiki/Cookbook:Spicy_Miso_Udon'}, {'title': 'Wonton Soup', 'url': 'https://en.wikibooks.org/wiki/Cookbook:Wonton_Soup'}]
📂 Opening Folder: Chinese recipes
{'name'

In [ ]:
import os
import json
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv()

def run_hosted_ingestion():
    # 1. Load your segregated JSONs
    loader = DirectoryLoader(
        './corpus_data',
        glob='**/*.json',
        loader_cls=JSONLoader,
        loader_kwargs={'jq_schema': '.content', 'text_content': True}
    )

    print("📦 Loading documents...")
    docs = loader.load()

    # 2. Bespoke Chunking
    # We use newlines as primary separators to keep recipe steps intact
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = text_splitter.split_documents(docs)
    print(f"✂️ Created {len(chunks)} chunks.")

    # 3. Hosted Vectorization
    # This sends chunks to HF servers instead of using your local CPU
    embeddings = HuggingFaceInferenceAPIEmbeddings(
        api_key=os.getenv("HF_API_KEY"), 
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    print("🧬 Fetching embeddings from Hosted API and building FAISS index...")
    # This might take a moment depending on network speed
    vector_db = FAISS.from_documents(chunks, embeddings)

    # 4. Save the Index
    # Even with hosted embeddings, we save the resulting vectors locally
    # so the demo doesn't need to re-call the API for every query.
    vector_db.save_local("faiss_index_hosted")
    print("✅ Ingestion Complete. Index saved to 'faiss_index_hosted'.")

run_hosted_ingestion()

In [2]:
import os
import json
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
loader = DirectoryLoader(
        './corpus_data',
        glob='**/*.json',
        loader_cls=JSONLoader,
        loader_kwargs={'jq_schema': '.content', 'text_content': True}
    )

print("📦 Loading documents...")
docs = loader.load()

📦 Loading documents...


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )
chunks = text_splitter.split_documents(docs)

In [9]:
embeddings = HuggingFaceInferenceAPIEmbeddings(
        api_key=os.getenv("HF_API_KEY"), 
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

/var/folders/4h/08pmbc590ds9q641w2wxcfbr0000gn/T/ipykernel_76114/1288271100.py:1: LangChainDeprecationWarning: The class `HuggingFaceInferenceAPIEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEndpointEmbeddings``.
  embeddings = HuggingFaceInferenceAPIEmbeddings(


In [10]:
type(docs[0])

langchain_core.documents.base.Document